# M4.9 · Nesne tespiti — IoU, NMS, YOLO, mAP, FPN

Sınıflandırma + konum belirlemeden (kutu regresyonu) çok nesneli tespite: IoU ölçütü, kayan pencere ve maksimum olmayanı bastırma (NMS), yoğun katmanları konvolüsyona çevirip ağı tam konvolüsyonel (FCN) yapmak ve tek geçişte ızgara tahmini, YOLO'nun hücre göreli koordinatları, k-means ile bulunan çapa kutuları ve log-ölçekli boyut tahmini, precision/recall eğrisinden 11 noktalı ve tüm noktalı AP, mAP@0.5 ve COCO'nun AP@[.50:.95]'i, FPN'nin yukarıdan-aşağı yolu ve RoI'nin piramit seviyesine atanması, Soft-NMS. Saf Python'da IoU/NMS/AP, NumPy'da döngüsüz IoU matrisi, Fast NMS ve IoU-uzaklıklı k-means, PyTorch'ta sentetik şekiller üzerinde sıfırdan küçük bir YOLO, sonda NMS varyantlarının ve sınıf-duyarlı "ofset hilesi"nin ölçümü.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m4-9-detection.html](https://m-gonul.github.io/pytorch-lab/m4-9-detection.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m4_9_detection.py
CHECKS = {
 "b": "import random\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"numpy kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n\n    def t_iou():\n        iou = need(ns, \"iou\")\n        close(iou((1, 1, 5, 5), (3, 2, 7, 6)), 6 / 26, what=\"IoU\")\n        close(iou((0, 0, 1, 1), (2, 2, 3, 3)), 0.0, what=\"ayrık kutular\")\n        close(iou((0, 0, 4, 4), (1, 1, 3, 3)), 0.25, what=\"iç içe\")\n        close(iou((0, 0, 2, 2), (0, 0, 2, 2)), 1.0, what=\"özdeş\")\n        B = [(0, 0, 10, 10), (1, 1, 11, 11), (20, 20, 30, 30), (8, 8, 18, 18), (21, 19, 31, 29)]\n        ensure(need(ns, \"nms\")(B, [0.9, 0.8, 0.7, 0.6, 0.5], 0.5) == [0, 2, 3], \"NMS örneği\")\n        ensure(ns[\"nms\"](B, [0.5, 0.8, 0.7, 0.6, 0.9], 0.5) == [4, 1, 3], \"skora göre sıralı\")\n    R_.test(\"IoU ve NMS\", t_iou)\n\n    def t_ap():\n        prec, rec = need(ns, \"pr_curve\")([1, 1, 0, 1, 0, 0, 1, 0], 5)\n        close(prec[3], 0.75, what=\"precision@4\")\n        close(need(ns, \"ap11\")(prec, rec), (6.5 + 8 / 7) / 11, what=\"11 noktalı AP\")\n        close(need(ns, \"ap_all\")(prec, rec), 0.2 * (2.75 + 4 / 7), what=\"tüm noktalı AP\")\n        gts = [(0, 0, 10, 10), (20, 0, 30, 10)]\n        dets = [(0, 0, 10, 10), (1, 0, 11, 10), (20, 0, 30, 10), (50, 50, 60, 60)]\n        ensure(need(ns, \"match\")(dets, [0.9, 0.8, 0.7, 0.6], gts, 0.5) == [True, False, True, False], \"yinelenen tespit yanlış\")\n        ensure(ns[\"match\"](dets, [0.6, 0.9, 0.7, 0.8], gts, 0.5) == [True, False, True, False], \"skor sırası: (1,0,11,10) önce eşleşir\")\n        rng = random.Random(0)\n        rows = []\n        for name, tp in ((\"mükemmel sıralama\", [1] * 6 + [0] * 6), (\"rastgele sıralama\", [rng.random() < 0.5 for _ in range(12)]), (\"ters sıralama\", [0] * 6 + [1] * 6)):\n            p, r = ns[\"pr_curve\"](tp, sum(tp))\n            rows.append([name, f\"{ns['ap11'](p, r):.3f}\", f\"{ns['ap_all'](p, r):.3f}\"])\n        R_.table(\"Aynı 12 tespit, farklı skor sıralaması\", [\"sıralama\", \"AP (11 nokta)\", \"AP (tüm noktalar)\"], rows,\n                 note=\"AP yalnız 'doğru tespitler yanlışlardan önce mi geliyor' sorusuna bakar: skorların mutlak değeri değil, sıralaması önemli. Doğru tespit sayısı aynı olsa da sıralama kötüleştikçe AP düşer; 11 noktalı biçim kaba bir yaklaşıklıktır.\")\n    R_.test(\"eşleştirme ve AP\", t_ap)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef _ref_nms(b, s, thr):\n    def iou(a, c):\n        iw = max(0, min(a[2], c[2]) - max(a[0], c[0])); ih = max(0, min(a[3], c[3]) - max(a[1], c[1]))\n        i = iw * ih\n        return i / ((a[2] - a[0]) * (a[3] - a[1]) + (c[2] - c[0]) * (c[3] - c[1]) - i)\n    keep = []\n    for i in np.argsort(-s, kind=\"stable\"):\n        if all(iou(b[i], b[j]) <= thr for j in keep):\n            keep.append(int(i))\n    return keep\n\n\ndef _boxes(rng, n):\n    c = rng.uniform(0, 100, (n, 2)); wh = rng.uniform(5, 30, (n, 2))\n    return np.concatenate([c - wh / 2, c + wh / 2], 1)\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngüler\", lambda: ([max_loops(src, 0, func=f) for f in (\"iou_matrix\", \"fast_nms\", \"ap_all\")], max_loops(src, 1, func=\"nms\"), max_loops(src, 1, func=\"anchor_kmeans\")))\n    rng = np.random.default_rng(0)\n\n    def t_nms():\n        A, B = _boxes(rng, 7), _boxes(rng, 5)\n        M = need(ns, \"iou_matrix\")(A, B)\n        ensure(M.shape == (7, 5) and (M >= 0).all() and (M <= 1).all(), \"IoU matrisi\")\n        close(np.diag(ns[\"iou_matrix\"](A, A)), np.ones(7), what=\"köşegen 1\")\n        for seed in range(5):\n            r = np.random.default_rng(seed)\n            b, s = _boxes(r, 120), r.random(120)\n            g = list(need(ns, \"nms\")(b, s, 0.4))\n            ensure(g == _ref_nms(b, s, 0.4), \"açgözlü NMS referansla aynı\")\n            f = set(need(ns, \"fast_nms\")(b, s, 0.4).tolist())\n            ensure(f <= set(g), \"Fast NMS ⊆ açgözlü NMS\")\n        b, s = _boxes(rng, 400), rng.random(400)\n        R_.note(f\"400 rastgele kutu, eşik 0.4: açgözlü NMS {len(ns['nms'](b, s, .4))}, Fast NMS {len(ns['fast_nms'](b, s, .4))} kutu tutuyor — Fast NMS bastırılmış kutuların da bastırmasına izin verdiği için biraz daha fazla siler; karşılığında tamamen paralel.\")\n    R_.test(\"NMS\", t_nms)\n\n    def t_ap():\n        close(need(ns, \"ap_all\")(np.array([.9, .8, .7, .6, .5, .4, .3, .2]), np.array([1, 1, 0, 1, 0, 0, 1, 0.]), 5), 0.2 * (2.75 + 4 / 7), what=\"AP\")\n        close(ns[\"ap_all\"](np.array([.2, .9, .4, .8, .5, .6, .3, .7]), np.array([0, 1, 0, 1, 0, 1, 1, 0.]), 5), 0.2 * (2.75 + 4 / 7), what=\"sırasız girdi\")\n    R_.test(\"AP\", t_ap)\n\n    def t_km():\n        r = np.random.default_rng(1)\n        ped = np.stack([r.normal(12, 1.5, 300), r.normal(32, 3, 300)], 1)\n        car = np.stack([r.normal(40, 4, 300), r.normal(18, 2, 300)], 1)\n        sq = np.stack([r.normal(20, 2, 300), r.normal(20, 2, 300)], 1)\n        wh = np.abs(np.concatenate([ped, car, sq]))\n        A, miou = need(ns, \"anchor_kmeans\")(wh, 3, 30, 0)\n        A = A[np.argsort(A[:, 0])]\n        close(A, np.array([[12, 32], [20, 20], [40, 18]]), atol=2.5, what=\"üç çapa: yaya, kare, araba\")\n        rows = []\n        for k in (1, 2, 3, 5, 9):\n            rows.append([k, f\"{ns['anchor_kmeans'](wh, k, 30, 0)[1]:.3f}\"])\n        R_.table(\"Çapa sayısı k ve ortalama en iyi IoU\", [\"k\", \"ort. IoU\"], rows,\n                 note=\"YOLOv2 Tablo 1'in küçük hâli: çapa sayısı arttıkça kutuların en yakın çapaya IoU'su artar, ama her çapa ağın çıkışına 5 + C sayı ekler. Öklid uzaklığı büyük kutuları kayırırdı; 1 − IoU ölçekten bağımsızdır.\")\n    R_.test(\"çapa k-means\", t_km)\n",
 "d": "import time\nimport torch\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    anchors = torch.tensor([[11.0, 11.0], [21.0, 21.0]])\n    X, T = need(ns, \"shapes_dataset\")(1500, 0)\n    Xt, Tt = ns[\"shapes_dataset\"](300, 1)\n\n    def t_code():\n        a = torch.tensor([[0., 0, 10, 10], [5, 5, 15, 15]])\n        close(need(ns, \"box_iou\")(a, a).numpy(), torch.tensor([[1, 25 / 175], [25 / 175, 1]]).numpy(), atol=1e-6, what=\"box_iou\")\n        for k in range(20):\n            b, l = T[k]\n            t = need(ns, \"encode\")(b, l, 8, anchors, 3)\n            ensure(int(t[..., 4].sum()) == len(b), \"her nesne tam bir çapaya\")\n            p = torch.zeros_like(t)\n            p[..., :2] = torch.logit(t[..., :2].clamp(1e-6, 1 - 1e-6))\n            p[..., 2:4] = t[..., 2:4]\n            p[..., 4] = torch.where(t[..., 4] > 0, 20.0, -20.0)\n            p[..., 5:] = 20 * t[..., 5:]\n            db, ds, dl = need(ns, \"decode\")(p, anchors, 64, 0.5)\n            o = torch.argsort(db[:, 0] * 1000 + db[:, 1])\n            g = torch.argsort(b[:, 0] * 1000 + b[:, 1])\n            close(db[o].numpy(), b[g].numpy(), atol=1e-3, what=\"decode(encode) = kutular\")\n            ensure(torch.equal(dl[o], l[g]), \"etiketler\")\n        b = torch.tensor([[0., 0, 10, 10], [1, 1, 11, 11], [20, 20, 30, 30], [8, 8, 18, 18], [21, 19, 31, 29]])\n        ensure(need(ns, \"nms\")(b, torch.tensor([.9, .8, .7, .6, .5]), 0.5).tolist() == [0, 2, 3], \"NMS\")\n    R_.test(\"kodlama / çözme / NMS\", t_code)\n\n    def t_train():\n        torch.manual_seed(0)\n        m = need(ns, \"TinyYOLO\")(2, 3)\n        ensure(tuple(m(X[:2]).shape) == (2, 8, 8, 2, 8), \"çıkış (N, S, S, A, 5 + C)\")\n        Y = torch.stack([ns[\"encode\"](b, l, 8, anchors, 3) for b, l in T])\n        opt = torch.optim.Adam(m.parameters(), 3e-3)\n        t0 = time.perf_counter()\n        hist = []\n        for ep in range(6):\n            m.train()\n            perm = torch.randperm(len(X))\n            for i in range(0, len(X), 64):\n                idx = perm[i:i + 64]\n                loss = need(ns, \"yolo_loss\")(m(X[idx]), Y[idx])\n                opt.zero_grad()\n                loss.backward()\n                opt.step()\n            hist.append(float(loss))\n        m.eval()\n        with torch.no_grad():\n            P = m(Xt)\n        rows = []\n        res = {}\n        for name, use_nms in ((\"NMS yok\", False), (\"NMS 0.5\", True)):\n            dets = []\n            for k in range(len(Xt)):\n                b, s, l = ns[\"decode\"](P[k], anchors, 64, 0.05)\n                if use_nms and len(b):\n                    keep = ns[\"nms\"](b, s, 0.5)\n                    b, s, l = b[keep], s[keep], l[keep]\n                dets.append((b, s, l))\n            a50 = need(ns, \"mean_ap\")(dets, Tt, 3, 0.5)\n            a75 = ns[\"mean_ap\"](dets, Tt, 3, 0.75)\n            coco = sum(ns[\"mean_ap\"](dets, Tt, 3, 0.5 + 0.05 * k) for k in range(10)) / 10\n            res[name] = a50\n            rows.append([name, f\"{sum(len(d[0]) for d in dets) / len(dets):.1f}\", f\"{a50:.3f}\", f\"{a75:.3f}\", f\"{coco:.3f}\"])\n        ensure(res[\"NMS 0.5\"] > 0.75, f\"mAP@0.5 = {res['NMS 0.5']:.3f} (> 0.75 beklenir)\")\n        R_.table(f\"TinyYOLO, 1500 görüntü × 6 epoch ({time.perf_counter() - t0:.0f} sn, kayıp {hist[0]:.2f} → {hist[-1]:.2f})\", [\"sonrası\", \"görüntü başı kutu\", \"mAP@0.5\", \"mAP@0.75\", \"AP@[.50:.95]\"], rows,\n                 note=\"Görüntü başına 1–3 nesne var. NMS'siz komşu hücre/çapa kopyaları yanlış pozitif olur. mAP@0.75 ve COCO ölçütü konum hassasiyetini cezalandırır: 8 × 8 ızgaradaki kaba dedektör 'nesneyi bulmakta' iyi, 'kutuyu tam oturtmakta' daha zayıf.\")\n    R_.test(\"eğitim ve mAP\", t_train)\n",
 "e": "import time\nimport torch\nfrom labcheck import need, ensure, fmt_time\n\n\ndef _boxes(g, n):\n    c = torch.rand(n, 2, generator=g) * 600\n    wh = torch.rand(n, 2, generator=g) * 60 + 10\n    return torch.cat([c - wh / 2, c + wh / 2], 1)\n\n\ndef check(ns, R_):\n    g = torch.Generator().manual_seed(0)\n\n    def t_eq():\n        b, s = _boxes(g, 300), torch.rand(300, generator=g)\n        lab = torch.randint(0, 4, (300,), generator=g)\n        keep = need(ns, \"nms_loop\")(b, s, 0.5)\n        ensure(set(need(ns, \"fast_nms\")(b, s, 0.5).tolist()) <= set(keep.tolist()), \"Fast NMS ⊆ açgözlü\")\n        ref = []\n        for c in range(4):\n            idx = torch.nonzero(lab == c)[:, 0]\n            ref += idx[ns[\"nms_loop\"](b[idx], s[idx], 0.5)].tolist()\n        ensure(sorted(need(ns, \"batched_nms\")(b, s, lab, 0.5).tolist()) == sorted(ref), \"ofset hilesi = sınıf başına NMS\")\n    R_.test(\"eşdeğerlikler\", t_eq)\n\n    def t_bench():\n        rows = []\n        for n in (500, 2000, 5000):\n            b, s = _boxes(g, n), torch.rand(n, generator=g)\n            r = [n]\n            for f in (ns[\"nms_loop\"], ns[\"fast_nms\"]):\n                best = 1e9\n                for _ in range(3):\n                    t0 = time.perf_counter()\n                    k = f(b, s, 0.5)\n                    best = min(best, time.perf_counter() - t0)\n                r += [fmt_time(best), len(k)]\n            rows.append(r)\n        R_.table(\"NMS süreleri (CPU, 600 × 600 sahnede rastgele kutular)\", [\"N\", \"açgözlü\", \"tutulan\", \"Fast NMS\", \"tutulan\"], rows,\n                 note=\"Açgözlü NMS tutulan kutu sayısı kadar sıralı adım atar (her adım küçük bir çekirdek çağrısı); Fast NMS tek bir N × N matris işlemidir: GPU'da paralel, ama bellek O(N²) ve biraz daha fazla kutu siler. Gerçek kütüphaneler (torchvision.ops.nms) açgözlü algoritmayı tek bir C++/CUDA çekirdeğinde bit maskeleriyle yapar.\")\n    R_.test(\"ölçüm\", t_bench)\n",
 "x1": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngüsüz\", lambda: [max_loops(src, 0, func=f) for f in (\"dense_to_conv\", \"fcn_apply\", \"sliding_dense\")])\n\n    def t():\n        rng = np.random.default_rng(0)\n        C, k, out = 10, 7, 20\n        W, b = rng.normal(size=(out, C * k * k)), rng.normal(size=out)\n        F7 = rng.normal(size=(C, 7, 7))\n        Wc = need(ns, \"dense_to_conv\")(W, C, k)\n        close(need(ns, \"fcn_apply\")(F7, Wc, b)[:, 0, 0], W @ F7.reshape(-1) + b, rtol=1e-10, what=\"7 × 7'de yoğun = konv. (1 × 1 çıkış)\")\n        F14 = rng.normal(size=(C, 14, 14))\n        y = ns[\"fcn_apply\"](F14, Wc, b)\n        ensure(y.shape == (out, 8, 8), \"14 × 14 → 8 × 8 ızgara\")\n        close(y, need(ns, \"sliding_dense\")(F14, W, b, k), rtol=1e-10, what=\"FCN = 64 konumda kayan yoğun katman\")\n        rows = []\n        for s in (224, 320, 448, 640):\n            g = s // 32 - 7 + 1\n            rows.append([f\"{s} × {s}\", f\"{s // 32} × {s // 32}\", f\"{g} × {g}\", g * g, f\"{(s / 224) ** 2:.1f}×\", f\"{g * g / (s / 224) ** 2:.1f}×\"])\n        R_.table(\"Toplam adımı 32 olan omurga: girdi boyutu → tahmin ızgarası\", [\"girdi\", \"darboğaz\", \"ızgara\", \"pencere\", \"FCN maliyeti (224'e göre)\", \"kayan pencereye göre kazanç\"], rows,\n                 note=\"Kayan pencere yaklaşımı ağı her pencere için baştan çalıştırır (maliyet ∝ pencere sayısı); FCN üst üste binen pencerelerin ortak hesabını paylaşır (maliyet ∝ görüntü alanı). 448'de 64 pencere 4× maliyete iner.\")\n    R_.test(\"FCN eşdeğerliği\", t)\n",
 "x2": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngüsüz\", lambda: (max_loops(src, 0, func=\"fpn_level\"), max_loops(src, 0, func=\"upsample2\")))\n\n    def t_lvl():\n        sizes = [16, 32, 64, 112, 160, 224, 320, 448, 900]\n        lv = need(ns, \"fpn_level\")(np.array(sizes), np.array(sizes))\n        ensure(list(lv) == [2, 2, 2, 3, 3, 4, 4, 5, 5], f\"seviyeler {list(lv)}\")\n        ensure(int(ns[\"fpn_level\"](100, 50)) == 2, \"100 × 50 → P2\")\n        R_.table(\"RoI boyutu → FPN seviyesi (k₀ = 4)\", [\"RoI kenarı\", \"seviye\", \"adım\"], [[s, f\"P{l}\", 2 ** l] for s, l in zip(sizes, lv)],\n                 note=\"RoI kenarı yarıya inince bir alt (daha ince) seviyeye geçilir: her seviyede nesne yaklaşık aynı sayıda öznitelik hücresi kaplar — görüntü piramidinin ölçek değişmezliği, ağın içinde ve tek geçişte.\")\n    R_.test(\"seviye ataması\", t_lvl)\n\n    def t_td():\n        rng = np.random.default_rng(0)\n        ch = [8, 16, 32, 64]\n        H = [32, 16, 8, 4]\n        d = 6\n        feats = [rng.normal(size=(c, h, h)) for c, h in zip(ch, H)]\n        lat = [rng.normal(size=(d, c)) / np.sqrt(c) for c in ch]\n        sm = [rng.normal(size=(d, d, 3, 3)) / 6 for _ in ch]\n        close(need(ns, \"upsample2\")(np.arange(4.).reshape(1, 2, 2))[0], np.array([[0, 0, 1, 1], [0, 0, 1, 1], [2, 2, 3, 3], [2, 2, 3, 3.]]), what=\"en yakın komşu\")\n        P = need(ns, \"top_down\")(feats, lat, sm)\n        ensure([p.shape for p in P] == [(d, h, h) for h in H], \"P2–P5 şekilleri\")\n        f2 = [f.copy() for f in feats]\n        f2[3][:, 0, 0] += 1.0\n        P2 = ns[\"top_down\"](f2, lat, sm)\n        ch_ = [np.abs(a - b).max(0) > 1e-12 for a, b in zip(P, P2)]\n        ensure(ch_[0][:6, :6].any() and not ch_[0][10:, 10:].any(), \"C5'teki tek hücre P2'de yalnız karşılık gelen 8 × 8 bölgeyi (+ 3×3 kenar) etkiler\")\n        f3 = [f.copy() for f in feats]\n        f3[0][:, 5, 5] += 1.0\n        P3 = ns[\"top_down\"](f3, lat, sm)\n        ensure(all(np.allclose(a, b) for a, b in zip(P[1:], P3[1:])), \"C2 değişikliği üst seviyelere sızmaz (yalnız yukarıdan aşağı akış)\")\n    R_.test(\"yukarıdan-aşağı yol\", t_td)\n",
 "x3": "import numpy as np\nfrom labcheck import close, need, ensure\n\n\ndef _hard(b, s, thr, iou_m):\n    order = np.argsort(-s, kind=\"stable\")\n    keep = []\n    while len(order):\n        i = order[0]\n        keep.append(i)\n        order = order[1:][iou_m(b[i][None], b[order[1:]])[0] <= thr]\n    return np.array(keep), s[keep]\n\n\ndef check(ns, R_):\n    iou_m = ns[\"iou_m\"]\n\n    def t_basic():\n        b = np.array([[0., 0, 10, 10], [1, 0, 11, 10], [50, 50, 60, 60]])\n        k, s = need(ns, \"soft_nms\")(b, np.array([.9, .8, .7]), 0.5)\n        o = 9 * 10 / (200 - 90)\n        ensure(list(k) == [0, 2, 1], f\"sıra {list(k)}\")\n        close(s, np.array([.9, .7, .8 * np.exp(-o ** 2 / .5)]), rtol=1e-9, what=\"Gauss azaltma\")\n        gts = [np.array([[0., 0, 10, 10], [20, 0, 30, 10]])]\n        d = [(np.array([[0., 0, 10, 10], [1, 0, 11, 10], [20, 0, 30, 10]]), np.array([.9, .8, .7]))]\n        close(need(ns, \"match_ap\")(d, gts, 0.5), 0.5 * 1 + 0.5 * 2 / 3, what=\"AP\")\n    R_.test(\"Soft-NMS ve AP\", t_basic)\n\n    def t_crowd():\n        rng = np.random.default_rng(0)\n        gts, raw = [], []\n        for _ in range(200):\n            c = rng.uniform(20, 80, 2)\n            w = rng.uniform(20, 30)\n            g1 = np.array([c[0] - w / 2, c[1] - w / 2, c[0] + w / 2, c[1] + w / 2])\n            sh = w * rng.uniform(0.15, 0.3)\n            g = np.stack([g1, g1 + np.array([sh, sh * 0.5, sh, sh * 0.5])])\n            gts.append(g)\n            b = np.concatenate([g + rng.normal(0, 1.0, g.shape), g[[0, 0, 1]] + rng.normal(0, 2.5, (3, 4))])\n            s = np.array([rng.uniform(0.85, 1), rng.uniform(0.5, 0.65), rng.uniform(0.7, 0.85), rng.uniform(0.7, 0.85), rng.uniform(0.3, 0.45)])\n            raw.append((b, s))\n        thrs = 0.5 + 0.05 * np.arange(10)\n        res, rows = {}, []\n        for name, f in ((\"NMS yok\", lambda b, s: (np.arange(len(s)), s)), (\"NMS 0.5\", lambda b, s: _hard(b, s, .5, iou_m)),\n                        (\"NMS 0.7\", lambda b, s: _hard(b, s, .7, iou_m)), (\"Soft-NMS σ = 0.5\", ns[\"soft_nms\"])):\n            d = [(b[k], s2) for (b, s) in raw for k, s2 in [f(b, s)]]\n            a = [need(ns, \"match_ap\")(d, gts, t) for t in thrs]\n            res[name] = a[0]\n            rows.append([name, f\"{a[0]:.3f}\", f\"{a[5]:.3f}\", f\"{np.mean(a):.3f}\"])\n        ensure(res[\"Soft-NMS σ = 0.5\"] > res[\"NMS 0.5\"] + 0.1 and res[\"Soft-NMS σ = 0.5\"] > res[\"NMS yok\"], \"Soft-NMS kalabalıkta kazanmalı\")\n        R_.table(\"Kalabalık çiftler (gerçek kutular arası IoU ≈ 0.52), 200 sahne\", [\"son işlem\", \"AP@0.5\", \"AP@0.75\", \"AP@[.50:.95]\"], rows,\n                 note=\"Arkadaki nesnenin tespiti öndekiyle IoU ≈ 0.5 olduğu için NMS 0.5 onu siler (recall kaybı); eşik 0.7 kopyaları kaçırır. Soft-NMS silmez, skoru IoU ile sürekli düşürür: kopyalar (IoU yüksek) dibe iner, komşu nesne (IoU orta) sıralamada kalır.\")\n    R_.test(\"kalabalık sahne\", t_crowd)\n"
}

In [ ]:
import importlib, labcheck, checks_m4_9_detection as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**Sınıflandırma + konum:** GAP katmanının üstüne ikinci bir 4 birimli çıkış (merkez x, y, genişlik, yükseklik; [0, 1]'e normalize, genişlik/yükseklik çoğu zaman karekökle) eklenir ve MSE ile eğitilir. Değerlendirmede MSE yerine **IoU**: kesişim alanı / birleşim alanı.

**Çok nesne:** eski yaklaşım tek nesne sınıflandırıcısını görüntü üzerinde farklı boyutlarda kaydırmaktır. Aynı nesne komşu konumlarda tekrar tekrar bulunur → **NMS**: (1) nesnellik (sigmoid + BCE) eşiğinin altını at, (2) en yüksek skorlu kutuyu al, onunla IoU'su eşiği (ör. 0.6) aşanları sil, (3) kalanlarla tekrarla.

**FCN (Long ve ark. 2015):** 100 × 7 × 7 öznitelik üzerindeki 200 nöronlu yoğun katman, 200 adet 7 × 7 "valid" filtreyle **aynı** hesaptır; konvolüsyon ise her girdi boyutunda çalışır. 224'te eğitilen ağa 448 verilirse darboğaz 14 × 14 olur, çıkış 8 × 8 ızgara — ağı 64 konumda kaydırmakla eşdeğer ama tek geçiş: **You Only Look Once**.

**YOLOv3:** hücre başına 5 kutu (her biri 4 koordinat + nesnellik) ve 20 sınıf olasılığı = 45 sayı. Merkez, hücreye göre sigmoid ofset ((0, 0) sol üst, (1, 1) sağ alt); her hücre yalnız merkezi kendi içinde olan kutulardan sorumlu. **Çapa kutuları**: eğitim kutularının (w, h)'sine k-means; ağ çapa başına $\ln$ ölçek faktörlerini tahmin eder (100 × 50 çapa, 1.5 ve 0.9 → 150 × 45). Çok ölçekli eğitim (her birkaç batch'te 330–608 arası rastgele çözünürlük), atlama bağlantıları.

**Ölçüt:** precision $TP/(TP+FP)$, recall $TP/(TP+FN)$. **AP**: recall 0, 0.1, …, 1'de "en az bu recall'daki en yüksek precision"ların ortalaması; sınıflar üzerinden ortalama **mAP**. Tespitte bir tahmin, sınıfı doğru **ve** IoU ≥ 0.5 ise doğru → mAP@0.5 (VOC); COCO 0.50, 0.55, …, 0.95 eşiklerinde ortalar: AP@[.50:.95].

**FPN (Lin ve ark. 2017):** omurganın aşağıdan-yukarı hiyerarşisini, yukarıdan-aşağı (2× büyütme) bir yolla ve 1×1 yanal bağlantılarla (toplama) birleştirir; her ölçekte anlamca güçlü haritalar. Bir RoI, $k=\lfloor k_0+\log_2(\sqrt{wh}/224)\rfloor$ seviyesinden öznitelik alır.

## 1 · Elle

<!-- çözüm -->

IoU: kesişim $[3,5]\times[2,5]$ = 2·3 = 6; birleşim 16 + 16 − 6 = 26 → 0.2308.

NMS: 0.9'luk kutu kalır; (1,1,11,11) ile IoU = 81/119 = 0.68 → silinir. 0.7'lik kalır; (8,8,18,18) ilk kutuyla 4/196 → kalır; (21,19,31,29) ile 0.7'lik arasında 81/119 → silinir. **3 kutu.**

YOLO: 169 · (25 + 20) = 7 605. Merkez: (7 + σ(0)) · 32 = 7.5 · 32 = 240. Boyut: 150 × 45 = 6 750.

AP: precision / recall: 1/0.2, 1/0.4, 0.667/0.4, 0.75/0.6, 0.6/0.6, 0.5/0.6, 0.571/0.8, 0.5/0.8. Zarf: recall ≤ 0.4 → 1, ≤ 0.6 → 0.75, ≤ 0.8 → 4/7, 0.9 ve 1.0 → 0. 11 nokta: (5 · 1 + 2 · 0.75 + 2 · 4/7) / 11 = 0.6948. Tüm noktalar: 0.2 · (1 + 1 + 0.75 + 4/7) = 0.6643 (recall 1'e hiç ulaşılmadığı için son 0.2'lik dilim sıfır katkı).

FCN: 14 − 7 + 1 = 8. FPN: $\sqrt{5000}=70.7$, $\log_2(70.7/224)=-1.66$, $\lfloor 2.34\rfloor=2$.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**IoU ve NMS (kutular $(x_1,y_1,x_2,y_2)$)**

- `iou`: IoU((1, 1, 5, 5), (3, 2, 7, 6))
- `kept`: NMS (eşik 0.5): (0,0,10,10) 0.9 · (1,1,11,11) 0.8 · (20,20,30,30) 0.7 · (8,8,18,18) 0.6 · (21,19,31,29) 0.5 — kaç kutu kalır?

**YOLO**

- `nout`: YOLOv3 tarzı çıktı: 13 × 13 ızgara, hücre başına 5 kutu (4 koordinat + nesnellik) + 20 sınıf → toplam sayı
- `cx`: 416 × 416 görüntü, 13 × 13 ızgara, hücre (satır 4, sütun 7), $t_x=0$: kutu merkezinin x'i (piksel)
- `area`: Çapa 100 piksel yüksek, 50 geniş; ağ $t_h=\ln 1.5$, $t_w=\ln 0.9$ tahmin ediyor: kutu alanı

**AP — skora göre sıralı tespitler: D D Y D Y Y D Y (D = doğru), toplam 5 gerçek nesne**

- `ap11`: 11 noktalı AP (VOC 2007)
- `apall`: Tüm noktalı AP (VOC 2010+)

**FCN ve FPN**

- `grid`: 224'te 7 × 7 darboğaz veren FCN'e 448 × 448 girdi; çıkış katmanı 7 × 7 valid konv.: çıkış ızgarası kenarı
- `lvl`: FPN Denk. (1), $k_0=4$: 100 × 50'lik RoI hangi seviyeye ($P_k$, $k\in[2,5]$)?

In [ ]:
cevaplar = dict(
    iou=None,
    kept=None,
    nout=None,
    cx=None,
    area=None,
    ap11=None,
    apall=None,
    grid=None,
    lvl=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"iou":[0.23076923076923078,0.000001],"kept":[3,0],"nout":[7605,0],"cx":[240,0],"area":[6750,0],"ap11":[0.6948051948051948,0.000001],"apall":[0.6642857142857143,0.000001],"grid":[8,0],"lvl":[2,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python, kutular `(x1, y1, x2, y2)` demetleri.

- `iou(a, b)`.
- `nms(boxes, scores, thr)` → tutulan indeksler (skor sırasıyla): açgözlü NMS.
- `match(det_boxes, det_scores, gt_boxes, thr=0.5)` → skora göre azalan sırada her tespit için doğru/yanlış listesi; her tespit, henüz eşleşmemiş gerçek kutular arasında IoU'su en yüksek olanla eşleşir (IoU ≥ thr ise doğru; aynı gerçek kutu ikinci kez eşleşemez → yinelenen tespit yanlıştır).
- `pr_curve(tp, n_gt)` → `(precision, recall)` listeleri.
- `ap11(prec, rec)` ve `ap_all(prec, rec)` — 11 noktalı ve tüm noktalı (zarf altındaki alan). Recall düzeylerini `k / 10` ile üret (`0.1 * k` kayan nokta tuzağı: `0.1 * 6 > 0.6`).

In [ ]:
%%writefile stage_b.py
def iou(a, b):
    ...


def nms(boxes, scores, thr):
    ...


def match(det_boxes, det_scores, gt_boxes, thr=0.5):
    ...


def pr_curve(tp, n_gt):
    ...


def ap11(prec, rec):
    ...


def ap_all(prec, rec):
    ...


print(iou((1, 1, 5, 5), (3, 2, 7, 6)))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy, döngüsüz (aksi belirtilmedikçe). `boxes (N, 4)`.

- `iou_matrix(A, B)` → `(N, M)`.
- `nms(boxes, scores, thr)` — açgözlü NMS, en fazla bir döngü (IoU'yu dışarıda bir kez hesapla).
- `fast_nms(boxes, scores, thr)` — döngüsüz (YOLACT): skora göre sırala, IoU matrisinin üst üçgenini al, her kutunun kendinden yüksek skorlu herhangi bir kutuyla en yüksek IoU'su eşiği aşıyorsa sil. Bastırılmış kutular da bastırabildiği için tuttuğu küme açgözlü NMS'ninkinin alt kümesidir.
- `ap_all(scores, tp, n_gt)` — sıralama, `cumsum`, `maximum.accumulate` ile zarf.
- `anchor_kmeans(wh, k, iters=30, seed=0)` — YOLOv2: (w, h) çiftlerini uzaklık $1-\text{IoU}$ ile kümele (kutuları aynı merkeze koyarak IoU); başlangıç: rastgele k örnek; tek döngü (iterasyonlar). Çapalar `(k, 2)` ve ortalama en iyi IoU.

In [ ]:
%%writefile stage_c.py
import numpy as np


def iou_matrix(A, B):
    ...


def nms(boxes, scores, thr):
    ...


def fast_nms(boxes, scores, thr):
    ...


def ap_all(scores, tp, n_gt):
    ...


def anchor_kmeans(wh, k, iters=30, seed=0):
    ...

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

PyTorch, sıfırdan küçük bir YOLO. Görüntüler 64 × 64, 3 sınıf (kare, daire, eşkenar dörtgen), ızgara S = 8 (hücre 8 piksel), 2 çapa.

- `shapes_dataset(n, seed)` → `(X (n,1,64,64), targets)`; hazır (verilir).
- `box_iou(a, b)` — `(N, 4) × (M, 4) → (N, M)`.
- `encode(boxes, labels, S, anchors, C, img=64)` → `(S, S, A, 5 + C)`: merkezin düştüğü hücre; şekil IoU'su en yüksek çapa; `[ofset_x, ofset_y, ln(w/w_a), ln(h/h_a), 1, one-hot]`.
- `decode(p, anchors, img=64, conf=0.5)` — tek görüntü çıkışı `(S, S, A, 5 + C)` → `(kutular, skor = nesnellik × sınıf olasılığı, etiket)`.
- `nms(boxes, scores, thr)`.
- `TinyYOLO(A=2, C=3)` — Conv-BN-LeakyReLU, üç 2× havuz, 1×1 başlık; çıkış `(N, S, S, A, 5 + C)`.
- `yolo_loss(p, t, l_coord=5, l_noobj=0.5)` — sorumlu çapalarda σ(ofset) ve log-ölçek MSE (λ_coord), nesnellik BCE (nesnesiz çapalarda λ_noobj), sınıf CE; batch'e bölünür.
- `mean_ap(dets, gts, C, thr=0.5)` — her görüntü için `(kutular, skorlar, etiketler)`; sınıf başına tüm noktalı AP'lerin ortalaması.

In [ ]:
%%writefile stage_d.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


def shapes_dataset(n, seed, size=64):
    g = torch.Generator().manual_seed(seed)
    yy, xx = torch.meshgrid(torch.arange(size) + 0.5, torch.arange(size) + 0.5, indexing="ij")
    imgs = torch.zeros(n, 1, size, size)
    targets = []
    for i in range(n):
        k = int(torch.randint(1, 4, (1,), generator=g))
        boxes, labels, tries = [], [], 0
        while len(boxes) < k and tries < 50:
            tries += 1
            s = float(torch.randint(8, 25, (1,), generator=g))
            cx = float(torch.rand(1, generator=g)) * (size - s) + s / 2
            cy = float(torch.rand(1, generator=g)) * (size - s) + s / 2
            b = [cx - s / 2, cy - s / 2, cx + s / 2, cy + s / 2]
            if any(not (b[2] <= o[0] or o[2] <= b[0] or b[3] <= o[1] or o[3] <= b[1]) for o in boxes):
                continue
            c = int(torch.randint(0, 3, (1,), generator=g))
            if c == 0:
                m = (xx >= b[0]) & (xx < b[2]) & (yy >= b[1]) & (yy < b[3])
            elif c == 1:
                m = (xx - cx) ** 2 + (yy - cy) ** 2 <= (s / 2) ** 2
            else:
                m = ((xx - cx).abs() + (yy - cy).abs()) <= s / 2
            imgs[i, 0][m] = 1.0
            boxes.append(b)
            labels.append(c)
        targets.append((torch.tensor(boxes), torch.tensor(labels)))
    imgs += 0.1 * torch.randn(imgs.shape, generator=g)
    return imgs, targets


def box_iou(a, b):
    ...


def encode(boxes, labels, S, anchors, C, img=64):
    ...


def decode(p, anchors, img=64, conf=0.5):
    ...


def nms(boxes, scores, thr):
    ...


class TinyYOLO(nn.Module):
    def __init__(self, A=2, C=3):
        super().__init__()
        ...

    def forward(self, x):
        ...


def yolo_loss(p, t, l_coord=5.0, l_noobj=0.5):
    ...


def mean_ap(dets, gts, C, thr=0.5):
    ...

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

- `nms_loop(boxes, scores, thr)` — her adımda yalnız seçilen kutunun IoU'sunu hesaplayan açgözlü NMS.
- `fast_nms(boxes, scores, thr)` — matris biçimi.
- `batched_nms(boxes, scores, labels, thr)` — sınıf-duyarlı NMS'yi tek çağrıda: her sınıfın kutularını `etiket × (max koordinat + 1)` kadar kaydır (farklı sınıflar asla kesişmez), sonra `nms_loop`.
- `bench(ns_)` → N ∈ {500, 2000, 5000} için süreler.

In [ ]:
%%writefile stage_e.py
import time
import torch


def box_iou(a, b):
    lt = torch.max(a[:, None, :2], b[None, :, :2])
    rb = torch.min(a[:, None, 2:], b[None, :, 2:])
    inter = (rb - lt).clamp(min=0).prod(-1)
    area = lambda x: (x[:, 2:] - x[:, :2]).prod(-1)
    return inter / (area(a)[:, None] + area(b)[None] - inter)


def nms_loop(boxes, scores, thr):
    ...


def fast_nms(boxes, scores, thr):
    ...


def batched_nms(boxes, scores, labels, thr):
    ...

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Yoğun katmanı konvolüsyona çevir: FCN = kayan pencere

NumPy. Öznitelik `F (C, H, W)`.

- `dense_to_conv(W, C, k)` — `W (out, C·k·k)` (C, k, k sırasıyla düzleştirilmiş) → `(out, C, k, k)`.
- `fcn_apply(F, Wc, b)` — valid konvolüsyon (`sliding_window_view` + `einsum`) → `(out, H − k + 1, W − k + 1)`.
- `sliding_dense(F, W, b, k)` — her k × k pencereyi düzleştirip yoğun katmana sok (aynı sonuç olmalı).

In [ ]:
%%writefile extra_x1.py
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view


def dense_to_conv(W, C, k):
    ...


def fcn_apply(F, Wc, b):
    ...


def sliding_dense(F, W, b, k):
    ...

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · FPN: yukarıdan-aşağı yol, yanal bağlantılar, RoI → seviye

NumPy.

- `fpn_level(w, h, k0=4, kmin=2, kmax=5)` — vektörel (dizi girdiler).
- `upsample2(x)` — en yakın komşu 2×.
- `top_down(feats, lat_w, smooth_w)` — `feats = [C2, C3, C4, C5]`, `C_i (c_i, H_i, W_i)`, her seviye bir öncekinin yarısı; `lat_w[i] (d, c_i)` 1×1 yanal; `smooth_w[i] (d, d, 3, 3)` "same" 3×3. $M_5=\text{lat}(C_5)$, $M_i=\text{up}(M_{i+1})+\text{lat}(C_i)$, $P_i=\text{smooth}(M_i)$. `[P2, …, P5]` döndür.

In [ ]:
%%writefile extra_x2.py
import numpy as np


def fpn_level(w, h, k0=4, kmin=2, kmax=5):
    ...


def upsample2(x):
    ...


def top_down(feats, lat_w, smooth_w):
    ...

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · Kalabalık sahne: Soft-NMS ve AP@[.50:.95]

NumPy (döngüler serbest).

- `soft_nms(boxes, scores, sigma=0.5, thresh=0.001)` — Gauss Soft-NMS: seçilen kutuyla IoU'su $o$ olan her kutunun skoru $s\leftarrow s\,e^{-o^2/\sigma}$; skoru `thresh` altına düşen atılır. `(indeksler, yeni skorlar)`.
- `match_ap(dets, gts, thr)` — çok görüntülü tüm noktalı AP; `dets = [(kutular, skorlar)]`, `gts = [kutular]` (tek sınıf).

In [ ]:
%%writefile extra_x3.py
import numpy as np


def iou_m(A, B):
    lt = np.maximum(A[:, None, :2], B[None, :, :2])
    rb = np.minimum(A[:, None, 2:], B[None, :, 2:])
    inter = np.clip(rb - lt, 0, None).prod(-1)
    ar = lambda X: (X[:, 2:] - X[:, :2]).prod(-1)
    return inter / (ar(A)[:, None] + ar(B)[None] - inter)


def soft_nms(boxes, scores, sigma=0.5, thresh=0.001):
    ...


def match_ap(dets, gts, thr=0.5):
    ...

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: Redmon ve ark. (2016): YOLOv1'i sıfırdan

[Redmon, Divvala, Girshick, Farhadi — You Only Look Once: Unified, Real-Time Object Detection (CVPR 2016)](https://arxiv.org/abs/1506.02640)

Makaledeki 7 × 7 × (2·5 + 20) çıkışlı YOLOv1'i, kaybı Denk. (3)'teki gibi (λ_coord = 5, λ_noobj = 0.5, karekök genişlik/yükseklik, "sorumlu" kutu = en yüksek IoU) uygula; PASCAL VOC'un küçük bir alt kümesinde (ör. 3 sınıf) önceden eğitilmiş bir ResNet-18 omurgasıyla eğit, mAP@0.5'i d aşamasındaki ölçücüyle raporla. Sonra çapa kutulu (YOLOv2) hâline geçip farkı ölç.


**Adımlar**

1. d aşamasındaki encode/decode/yolo_loss'u v1 kodlamasına uyarla (çapa yok, hücre başına B kutu).
2. Hata analizi: makale Şekil 4'teki gibi hataları konum / benzer sınıf / arka plan olarak sınıfla.

**Kabul ölçütleri**

- Eğitim kaybı düşer, doğrulama mAP@0.5 sıfırdan belirgin büyük.
- Hata dağılımı tablosu; YOLOv1'in konum hatası baskın çıkmalı (makaleyle uyumlu).

## Makale ödevi: Lin ve ark. (2017): Öznitelik piramidi ağları

[Lin, Dollár, Girshick, He, Hariharan, Belongie — Feature Pyramid Networks for Object Detection (CVPR 2017)](https://arxiv.org/abs/1612.03144)

d aşamasındaki tek ölçekli dedektörü, nesne boyutlarının 4–48 piksel arasında değiştiği bir sentetik veri setinde (64 → 128 tuval) dene; ardından x2'deki yukarıdan-aşağı yol ve yanal bağlantılarla P3–P5 çıkışlı bir FPN başı ekle, her seviyeye uygun çapaları ata. Makale Tablo 1'deki ablasyonun küçük sürümü: (a) yalnız en kaba seviye, (b) yukarıdan-aşağı yolsuz piramit (SSD tarzı), (c) yanal bağlantısız, (d) tam FPN — küçük/orta/büyük nesnelerde AP.


**Adımlar**

1. Ölçek başına AP için x3'teki ölçücüyü boyut aralıklarına göre filtrele.

**Kabul ölçütleri**

- Tam FPN küçük nesnelerde AP'yi belirgin artırır.
- Yanal bağlantısız sürüm konumlandırmada (AP@0.75) geride kalır.

## Kaynaklar

- [Géron — Hands-On Machine Learning (3. baskı), Bölüm 14](https://github.com/ageron/handson-ml3) — Apache-2.0 (kod)
- [Redmon ve ark. — You Only Look Once (CVPR 2016)](https://arxiv.org/abs/1506.02640) — arXiv
- [Redmon & Farhadi — YOLO9000 (CVPR 2017) / YOLOv3 (2018)](https://arxiv.org/abs/1612.08242) — arXiv
- [Lin ve ark. — Feature Pyramid Networks for Object Detection (CVPR 2017)](https://arxiv.org/abs/1612.03144) — arXiv
- [Long, Shelhamer, Darrell — Fully Convolutional Networks for Semantic Segmentation (CVPR 2015)](https://arxiv.org/abs/1411.4038) — arXiv
- [Bodla ve ark. — Soft-NMS (ICCV 2017)](https://arxiv.org/abs/1704.04503) — arXiv
- [Bolya ve ark. — YOLACT (Fast NMS), ICCV 2019](https://arxiv.org/abs/1904.02689) — arXiv
- [Everingham ve ark. — The PASCAL VOC Challenge (IJCV 2010)](http://host.robots.ox.ac.uk/pascal/VOC/) — ©
- [CS231n — Detection and Segmentation](https://cs231n.stanford.edu/) — ©